## Endpoints and EndpointSlices — what the Service actually points at

The Service holds the *selector*. The *resolved list of pod IPs* lives in a separate object:

- **`Endpoints`** — the original kind, one per Service, same name. Lists every ready pod's IP + port.
- **`EndpointSlice`** — the modern kind (default since 1.21). Up to 100 endpoints per slice; big Services get several. Lower overhead at scale.

Both are filled in automatically by the **endpoints controller**, which updates them when a matching pod becomes Ready, stops being Ready, is deleted, or when the selector changes. `kubectl get endpoints api` reads the live answer to "which pod IPs is this Service pointing at right now?"

**The single most common Services bug** is a Service showing `<none>` endpoints — meaning **no ready pods match the selector.** Almost always:

- selector and pod labels disagree (a typo),
- pods aren't ready (failing probe, still pulling), or
- pods are in a different namespace from the Service.

`kubectl describe service <name>` shows the selector and the endpoint addresses side by side — the first stop when something can't connect.

### A Service with no selector

Write a Service with **no selector** and hand-author its `Endpoints` to point at an external IP:

```yaml
kind: Service
metadata: { name: external-db }
spec: { ports: [{ port: 5432 }] }
---
kind: Endpoints
metadata: { name: external-db }   # same name as the Service
subsets:
  - addresses: [{ ip: 10.0.5.20 }]
    ports: [{ port: 5432 }]
```

This gives an external system (a managed RDS, a legacy VM) a **Kubernetes-shaped name** — clients call `external-db`, and you can later swap in an in-cluster Deployment without touching them. On our map, Endpoints are the live wire from the **Service** chip to the **Pods** it currently fronts.